# Training on Google Colab -- multi-label composite branch

This trains the **multi-label** model from branch `eileen-multilabel-complete` -- the one that
can flag more than one class per window at once (e.g. a real signal WITH a
jammer overlaid on top), instead of forcing exactly one winner. If you have
not read the branch's summary yet: the model output changed from softmax
(exactly one class) to sigmoid (each class judged independently), the
dataset now includes composite (jammer-overlaid-on-victim) examples in
addition to standalone ones, and every downstream script (train, evaluate,
ensemble, variance) was updated to match.

Colab is a **different computer** -- it cannot see your local drive. So this
notebook:

1. pulls the **code** in from GitHub
2. pulls the **data** in by upload (the small processed arrays, not raw
   RadioML/RadChar)
3. trains (ensemble + single model) and checks the >80% judged-class recall
   gate
4. pushes **everything** back out to your machine

## Read this before you start

**Colab's disk is temporary.** Everything under `/content/` is deleted when
the session ends -- idle timeout, or ~12 hours maximum. If you train for 20
minutes and close the tab without running the download cell, the model is
gone.

**Enable the GPU first:** Runtime -> Change runtime type -> Hardware
accelerator -> GPU. Do this *before* running anything; switching later
restarts the session and wipes your uploads.

**Run cells top to bottom, in order.** Sanity-check runs *before* any
training, and the dataset is never rebuilt inside Colab -- it is built
locally (where RadioML/RadChar live) and uploaded as three small arrays.


## 1. Get the code

**The branch must be pushed to GitHub first** -- Colab clones from GitHub,
it cannot see a branch that only exists on your local machine. If
`git push -u origin eileen-multilabel-complete` has not been run yet, do that before this
cell.

If the repo is private, cloning fails. Either make it public, or upload a
zip of the repo instead.


In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eileen-multilabel-complete https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pwd

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, matplotlib.
# Only these are missing:
!pip install -q pyyaml h5py

## 2. Check the GPU is actually attached

If this says `CUDA: False`, you skipped the Runtime -> Change runtime type
step. Training still works on CPU, just much slower.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. Get the data in

Upload the three arrays from your machine:

    data/processed/X.npy
    data/processed/y.npy
    data/processed/snr_labels.npy

Build these **locally, on this branch, with RadioML/RadChar available**:

    python -m src.data.build_dataset

**Important shape change from the old (single-label) dataset**: `y.npy` is
now `(N, 8)` -- multi-hot, one column per class, not `(N,)` with one integer
class index per row. If you upload arrays built on the old `main`/
`eavan-train-overall` branch, the sanity check below will fail loudly rather
than silently training on the wrong label format -- that is the point of
running it first.

**Use this cell OR the Drive cell below, not both.**


In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data/processed', exist_ok=True)
print('Select X.npy, y.npy and snr_labels.npy (you can pick all three at once)')
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'data/processed/{name}')
!ls -la data/processed/

### Alternative: mount Google Drive

Better if you will run this repeatedly -- the files persist between
sessions, so you upload once instead of every time. Put the arrays in a
`sedic/` folder in your Drive first.

**Commented out on purpose** -- this is an alternative to the upload cell
above, not an extra step. Uncomment and run *instead of* the upload cell,
not after it.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/processed
# !cp /content/drive/MyDrive/sedic/*.npy data/processed/

## 4. Sanity check -- BEFORE spending any GPU time

Confirms the code runs and the data loaded in the right (multi-label) shape.
Takes seconds. **If this fails, stop -- do not run the training cells
below.**


In [ ]:
!python -m pytest -q

import numpy as np
X = np.load('data/processed/X.npy')
y = np.load('data/processed/y.npy')
print('X:', X.shape, X.dtype)
print('y:', y.shape, y.dtype, '(expect (N, 8) -- multi-hot, NOT (N,))')
assert y.ndim == 2 and y.shape[1] == 8, (
    'y.npy looks like the OLD single-label format -- rebuild the dataset '
    'on this branch (python -m src.data.build_dataset) and re-upload.'
)

from src.config import CLASSES
n_composite = int((y.sum(axis=1) > 1).sum())
print(f'composite (>1 class present) windows: {n_composite} / {len(y)}')
for i, c in enumerate(CLASSES):
    print(f'  {c:<12} {int(y[:, i].sum()):>6}  present')

## 5. Train

Same three-run pattern as before, all updated for multi-label:

1. **Ensemble** (`train_ensemble.py`) -- 5 seeds, sigmoid outputs averaged.
   This is your main result: seed variance on this project has been
   measured up to ~10 points on jamming recall alone, so a single run
   cannot reliably answer "is it 80%" on its own.
2. **Variance measurement** -- quantifies that swing directly. Prints only,
   no file is written, so **copy the printed spread numbers somewhere
   before you move on**.
3. **Single model** (`src.train`) -- a plain baseline run, useful to quote
   alongside the ensemble result in the brief.

Watch `val_loss`/`val_bit_acc` in the single-model run: falling loss means
learning; `val_bit_acc` is per-class-bit accuracy now, not "did we guess the
one right class" (multiple classes can be true on the same window).


In [ ]:
!python scripts/train_ensemble.py --models 5

In [ ]:
!python scripts/measure_variance.py --runs 5

In [ ]:
!python -m src.train

In [ ]:
!python -m src.evaluate

### Is it 80%? -- the headline answer

Reads back the two scorecards just written and states PASS/FAIL plainly, so
you do not have to scan the printed logs above.


In [ ]:
import json

print('--- Single model (src/evaluate.py) ---')
sc = json.load(open('evals/scorecard.json'))
for cls, r in sc['benchmark']['judged_classes'].items():
    print(f"  {cls:<12} recall={r['recall']:.4f}  {'PASS' if r['passed'] else 'FAIL'}")
print(f"  OVERALL: {'PASS' if sc['benchmark']['passed'] else 'FAIL'}")

print()
print('--- Ensemble (scripts/train_ensemble.py) ---')
ens = json.load(open('evals/ensemble_scorecard.json'))
for cls, recall in ens['ensemble'].items():
    passed = recall >= 0.80
    print(f"  {cls:<12} recall={recall:.4f}  {'PASS' if passed else 'FAIL'}")
print(f"  OVERALL: {'PASS' if ens['passed'] else 'FAIL'}")
print()
print('The ensemble number is the one to trust -- it is the one that '
      'cancels seed-initialisation noise (see cell above).')

### Preview the plots before downloading

`diagnose_jamming.py` (jamming sub-type breakdown) is **not included here**
-- it still assumes the old single-label `argmax` output and has not been
updated for multi-label yet. It is a diagnostic-only script (not part of the
scored benchmark), so it is safe to skip for now.


In [ ]:
from IPython.display import Image, display
display(Image('evals/confusion_matrix.png'))
display(Image('evals/accuracy_vs_snr.png'))

## 6. Get everything out -- DO NOT SKIP THIS

This is the step people forget. Everything above is deleted when the
session ends. Save the downloads into `results/` and `evals/` in your local
repo.


In [ ]:
from google.colab import files
import os, glob

paths = [
    'results/best_model.pt',
    'evals/scorecard.json',
    'evals/confusion_matrix.png',
    'evals/accuracy_vs_snr.png',
    'evals/ensemble_scorecard.json',
] + sorted(glob.glob('results/ensemble_*.pt'))

for path in paths:
    if os.path.exists(path):
        files.download(path)
    else:
        print('missing:', path)

### Or save straight to Drive (survives the session)

In [ ]:
# !mkdir -p /content/drive/MyDrive/sedic/eavan-run-2026-08-23
# !cp results/best_model.pt results/ensemble_*.pt evals/*.json evals/*.png /content/drive/MyDrive/sedic/eavan-run-2026-08-23/